# `vla_control` tutorial

This notebook shows how to use the Python library from Jupyter to talk to the Unity simulator, inspect state, capture images, move the proxy, manage cameras, and run a simple rollout.

## Before you start

1. Open the Unity project in `unity/`.
2. Open `Assets/Scenes/Main.unity`.
3. Enter Play Mode so the local HTTP server starts.
4. From `python_controller/`, run `uv sync` if you have not installed dependencies yet.

The default Unity endpoint is `http://127.0.0.1:8080`.

In [ ]:
from pathlib import Path

from IPython.display import display

from vla_control import (
    ActionAdapter,
    DummyPolicy,
    EvaluationRunner,
    PoseCommand,
    RolloutConfig,
    UnityClient,
    UnityConfig,
    UpsertCameraRequest,
    Vector3,
    run_smoke_test,
)


## Create a client

`UnityConfig` uses `pydantic-settings`, so you can also override values with environment variables like `VLA_CONTROL_HOST` and `VLA_CONTROL_PORT`.

In [ ]:
config = UnityConfig()
client = UnityClient(config)

print(config.model_dump())
client.health()


## Reset and inspect simulator state

The Unity server reports both the current pose and the target pose, plus timing metadata like `physics_dt` and `steps_per_action`.

In [ ]:
client.reset()
state = client.get_state()
state


## Capture an image

`get_camera()` returns a `PIL.Image.Image`, so the notebook can display it directly.

In [ ]:
main_image = client.get_camera("main", width=256, height=256, quality=80)
display(main_image)
main_image.size


## Send a pose command

The Unity API uses an absolute world-frame pose plus a normalized gripper value.

Quaternion note: the identity rotation is `x=0, y=0, z=0, w=1`.

In [ ]:
command = PoseCommand.world(
    x=0.20,
    y=0.95,
    z=0.15,
    gripper=0.50,
)
command


In [ ]:
move_response = client.move_to_pose(command)
step_response = client.step_control_interval()
state_after_move = client.get_state()

move_response, step_response, state_after_move


## Manage named cameras

The camera API can list cameras, create runtime cameras, attach them to the proxy mount, and delete runtime-only cameras.

In [ ]:
camera_inventory = client.list_cameras()
camera_inventory


In [ ]:
wrist_request = UpsertCameraRequest.mounted(
    name="wrist",
    mount_target="proxy_camera_mount",
    local_position=Vector3(x=0.0, y=0.08, z=-0.12),
    local_rotation_euler=Vector3(x=15.0, y=0.0, z=0.0),
)
wrist_response = client.upsert_camera(wrist_request)
wrist_response


In [ ]:
display(client.get_camera("wrist", width=256, height=256, quality=80))


In [ ]:
client.list_cameras()


You can also create a free world camera instead of a mounted one.

In [ ]:
monitor_request = UpsertCameraRequest.world(
    name="monitor",
    world_position=Vector3(x=1.2, y=1.4, z=-1.2),
    world_rotation_euler=Vector3(x=35.0, y=135.0, z=0.0),
)
client.upsert_camera(monitor_request)


In [ ]:
display(client.get_camera("monitor", width=256, height=256, quality=80))


In [ ]:
client.delete_camera("wrist")
client.delete_camera("monitor")
client.list_cameras()


## Run the built-in smoke test

This is a lightweight end-to-end sanity check that saves one image if you provide an output directory.

In [ ]:
smoke_result = run_smoke_test(client, save_dir=Path("artifacts/notebook_smoke"))
smoke_result


## Run a dummy rollout

The synchronous runner keeps Python as the clock: capture image, read state, predict, adapt, send `move_to_pose`, then call `step_control_interval()`.

In [ ]:
runner = EvaluationRunner(client, DummyPolicy(), ActionAdapter())
runner


In [ ]:
rollout_config = RolloutConfig(
    instruction="move the proxy toward the dummy goal",
    max_steps=6,
    artifact_root=Path("artifacts/notebook_rollout"),
    run_name="demo",
)
summary = runner.run_rollout(rollout_config)
summary.status, summary.total_steps, summary.artifact_dir


In [ ]:
summary.records[0]


## Cleanup

Reset the simulator so the next notebook run starts from a known state.

In [ ]:
client.reset()
